In [1]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect(':memory:')

# ----------------------------------------------------
# 1. df_login：用户登录明细（故意制造同一天重复登录、日期格式群魔乱舞）
# ----------------------------------------------------
login_data = {
    'login_id':   [1,            2,            3,            4,            5,            6,            7,            8,            9],
    'user_id':    [801,          801,          801,          801,          801,          802,          802,          803,          803],
    # 👈 脏数据：包含前导空格、大小写混合的日期文本
    'login_date': ['2026-06-01', ' 2026-06-01', '2026-06-02', '2026-06-03', '2026-06-05', '2026-06-01', '2026-06-02', '2026-06-01', '2026-06-03']
}
df_login = pd.DataFrame(login_data)
df_login.to_sql('login_log', conn, index=False, if_exists='replace')

# ----------------------------------------------------
# 2. df_risk_users：风控白名单/灰产标记表
# ----------------------------------------------------
risk_data = {
    'user_id':   [801,       802,       803],
    'risk_level':[' HIGH ',  'Low',     'High'] # 👈 脏数据：前后空格、大小写不一
}
df_risk_users = pd.DataFrame(risk_data)
df_risk_users.to_sql('risk_users', conn, index=False, if_exists='replace')

print("====== 🛡️ 第四宇宙风控数仓已锁死 =======")
print("原始登录表 login_log：")
print(df_login)
print("\n原始风险表 risk_users：")
print(df_risk_users)

====== 🛡️ 第四宇宙风控数仓已锁死 =======
原始登录表 login_log：
   login_id  user_id   login_date
0         1      801   2026-06-01
1         2      801   2026-06-01
2         3      801   2026-06-02
3         4      801   2026-06-03
4         5      801   2026-06-05
5         6      802   2026-06-01
6         7      802   2026-06-02
7         8      803   2026-06-01
8         9      803   2026-06-03

原始风险表 risk_users：
   user_id risk_level
0      801      HIGH 
1      802        Low
2      803       High


📋 业务需求：
黑产团伙经常利用自动化脚本在电商平台“连续登录”来养号捞券。风控总监现在要求你：

找出所有风险等级为大写“HIGH”的危险用户，计算出他们在苹果/安卓全平台下，各自由于脚本刷单而产生的“最大连续登录天数（Max Consecutive Days）”。最终输出不带索引。

📊 输出字段要求：
最终报表必须精准对齐：
user_id | max_consecutive_days

In [13]:
# SQL轨道
sql_query = """
WITH risk_user_cleaned AS (
    SELECT user_id, UPPER(TRIM(risk_level)) AS risk_level
    FROM risk_users
),
high_risk_user AS (
    SELECT user_id FROM risk_user_cleaned WHERE risk_level = 'HIGH'
),

-- 1. 🔥 核心堵漏防御圈：在去重的同时，用 TRIM 物理超度所有 user_id 和日期的幽灵空格！
login_dedup AS (
    SELECT DISTINCT 
           TRIM(user_id) AS user_id, 
           TRIM(login_date) AS login_date
    FROM login_log
),

-- 2. 此时进场的 login_date 已经是纯净无瑕的 '2026-06-01'，julianday 绝不砸盘！
login_with_rnk AS (
    SELECT 
        d.user_id,
        d.login_date,
        CAST(julianday(d.login_date) AS INT) AS date_num,
        ROW_NUMBER() OVER(PARTITION BY d.user_id ORDER BY d.login_date ASC) AS rnk
    FROM login_dedup AS d
    INNER JOIN high_risk_user AS h ON d.user_id = h.user_id
),

-- 3. 时空平移相减
login_islands AS (
    SELECT 
        user_id,
        login_date,
        (date_num - rnk) AS base_day
    FROM login_with_rnk
),
-- 4. 第一级坍塌：算每段连续几天
consecutive_groups AS (
    SELECT 
        user_id,
        COUNT(login_date) AS consecutive_days
    FROM login_islands
    GROUP BY user_id, base_day
)
-- 5. 第二级终极收割：轰出最长连续天数
SELECT 
    user_id,
    MAX(consecutive_days) AS max_consecutive_days
FROM consecutive_groups
GROUP BY user_id
ORDER BY max_consecutive_days DESC;
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

  user_id  max_consecutive_days
0     801                     3
1     803                     1


In [ ]:
# ====================================================
# 🐼 PANDAS 轨道 - 完全体风控阻击链条
# ====================================================

# 1. 净化黑产高危灰名单
high_risk_user = (
    df_risk_users
    .assign(risk_level=lambda df: df['risk_level'].str.strip().str.upper())
    .query("risk_level == 'HIGH'")
)

# 2. 净化登录日志，打上绝对时间座次
df_date_rnk = (
    df_login
    .assign(
        user_id=lambda df: df['user_id'].astype(str).str.strip().astype(int),
        login_date=lambda df: df['login_date'].str.strip()
    )
    # 💣 拆弹微操：先洗净，后坍塌
    .drop_duplicates(subset=['user_id', 'login_date'], ignore_index=True)
    
    # 🔥 核心修正：在发号前，必须执行硬核时空整队（人按升序，时间按升序）！
    # 唯有如此，接下来的物理序号增长斜率才能与日历对齐！
    .sort_values(by=['user_id', 'login_date'], ascending=[True, True])
    
    # 🛡️ 翻阅 SYNTAX RULES：用 method='first' 镜像还原 ROW_NUMBER()
    .assign(rnk=lambda df: df.groupby('user_id')['login_date'].rank(method='first').astype(int))
)

# 3. 跨表合围，执行时空平移魔法
df_merge = (
    df_date_rnk
    .merge(high_risk_user, on='user_id', how='inner')
    # 🌟 坐标点 减去 刻度尺，平移出连续基准日
    .assign(base_date=lambda df: pd.to_datetime(df['login_date']) - pd.to_timedelta(df['rnk'], unit='D'))
)

# 4. 双层降维压榨，轰出最大连续天数
df_final = (
    df_merge
    # 第一层：压榨出每个人各段连续孤岛的天数
    .groupby(['user_id', 'base_date']).size()
    .reset_index(name='consecutive_days')
    # 第二层：点杀提取每个人这辈子最长的连续记录
    .groupby('user_id').agg(max_consecutive_days=('consecutive_days', 'max'))
    .reset_index()
    # 报表清洗：按天数倒序排列
    .sort_values(by='max_consecutive_days', ascending=False)
)

print("=== 🛡️ PANDAS 轨道完全体：黑产连续登录深度穿透报表 ===")
# 🔥 终极修正：切掉索引，无痕交付商业报表！
print(df_final.to_string(index=False))

   user_id  max_consecutive_days
0      801                     3
1      803                     1
